# DVCLive and DVC Monitoring for Default sklearn Experiment

This notebook illustrates how the default experiment in examples/sklearn exposes monitoring signals through DVCLive and DVC metadata.

What this notebook covers:
- inspect default monitoring configuration (dvclive_enabled and dvc_plugin)
- assemble a bounded, reproducible optimize command for a short demo run
- inspect monitoring artifacts and DVC metadata after a run

In [10]:
from __future__ import annotations

import json
import os
import shutil
import subprocess
from pathlib import Path

from hydra import compose, initialize_config_dir
from omegaconf import OmegaConf

cwd = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in [cwd, *cwd.parents] if (p / "deckard").exists() and (p / "examples").exists()), cwd)
NOTEBOOK_DIR = PROJECT_ROOT / "docs" / "notebooks"
CONFIG_DIR = PROJECT_ROOT / "examples" / "sklearn" / "config"
BUILD_DIR = NOTEBOOK_DIR / "build" / "dvclivc"
BUILD_DIR.mkdir(parents=True, exist_ok=True)

DECKARD_CMD = shutil.which("deckard")
DVC_CMD = shutil.which("dvc")

print(f"PROJECT_ROOT={PROJECT_ROOT}")
print(f"CONFIG_DIR={CONFIG_DIR}")
print(f"BUILD_DIR={BUILD_DIR}")
print(f"deckard_cli_found={DECKARD_CMD is not None}")
print(f"dvc_cli_found={DVC_CMD is not None}")

PROJECT_ROOT=/Users/c.meyers/Documents/deckard
CONFIG_DIR=/Users/c.meyers/Documents/deckard/examples/sklearn/config
BUILD_DIR=/Users/c.meyers/Documents/deckard/docs/notebooks/build/dvclivc
deckard_cli_found=True
dvc_cli_found=True


## 1) Inspect Default Monitoring Configuration

In [11]:
with initialize_config_dir(version_base="1.3", config_dir=str(CONFIG_DIR)):
    cfg = compose(config_name="default", overrides=["score=classification", "+stage=score"] )

dvc_plugin_cfg = OmegaConf.to_container(cfg.get("dvc_plugin"), resolve=False, throw_on_missing=False)
if not isinstance(dvc_plugin_cfg, dict):
    dvc_plugin_cfg = {}

monitoring_view = {
    "dvclive_enabled": bool(cfg.get("dvclive_enabled", False)),
    "dvc_plugin": dvc_plugin_cfg,
    "hydra_run_dir": OmegaConf.select(cfg, "hydra.run.dir"),
    "hydra_sweep_dir": OmegaConf.select(cfg, "hydra.sweep.dir"),
    "optimizers": list(cfg.get("optimizers", [])),
    "directions": list(cfg.get("directions", [])),
}

print(json.dumps(monitoring_view, indent=2, sort_keys=True, default=str))
assert monitoring_view["dvclive_enabled"] is True

{
  "directions": [
    "maximize",
    "maximize",
    "maximize"
  ],
  "dvc_plugin": {
    "cache_images": false,
    "dvclive_dir": null,
    "enabled": "${dvclive_enabled}",
    "make_report": true,
    "make_summary": true,
    "monitor_system": false,
    "report_mode": "html",
    "resume": true,
    "save_dvc_exp": false
  },
  "dvclive_enabled": true,
  "hydra_run_dir": null,
  "hydra_sweep_dir": null,
  "optimizers": [
    "accuracy",
    "evasion_accuracy",
    "attack_generation_time"
  ]
}


## 2) Prepare a Bounded Demo Command

The command below keeps the run intentionally small while still exercising monitoring behavior.

In [12]:
study_name = "dvclivc_demo"
storage_uri = f"sqlite:///{(BUILD_DIR / 'optuna.db').as_posix()}"
dvclive_dir = (BUILD_DIR / "dvclive").as_posix()

demo_cmd = [
    DECKARD_CMD or "deckard",
    "optimize",
    "--multirun",
    "--config-name",
    "default",
    "score=classification",
    "+stage=score",
    f"hydra.sweeper.study_name={study_name}",
    f"hydra.sweeper.storage={storage_uri}",
    "hydra.sweeper.n_trials=1",
    "hydra.sweeper.n_jobs=1",
    "pruning_enabled=false",
    "dvclive_enabled=true",
    "dvc_plugin.monitor_system=true",
    f"+dvclive_dir={dvclive_dir}",
    f"hydra.sweep.dir={(BUILD_DIR / 'outputs').as_posix()}",
    "hydra.sweep.subdir=${hydra.job.num}",
]

print("Demo command (system monitoring enabled):")
print(" ".join(demo_cmd))

Demo command (system monitoring enabled):
/Users/c.meyers/Documents/deckard/.venv/bin/deckard optimize --multirun --config-name default score=classification +stage=score hydra.sweeper.study_name=dvclivc_demo hydra.sweeper.storage=sqlite:////Users/c.meyers/Documents/deckard/docs/notebooks/build/dvclivc/optuna.db hydra.sweeper.n_trials=1 hydra.sweeper.n_jobs=1 pruning_enabled=false dvclive_enabled=true dvc_plugin.monitor_system=true +dvclive_dir=/Users/c.meyers/Documents/deckard/docs/notebooks/build/dvclivc/dvclive hydra.sweep.dir=/Users/c.meyers/Documents/deckard/docs/notebooks/build/dvclivc/outputs hydra.sweep.subdir=${hydra.job.num}


## 3) Inspect Monitoring Artifacts

Run this section after setting run_demo=True and executing the previous cell.

### 3a) Optional Demo Execution and Runtime Monitoring

By default the cell below does not execute the sweep.
Set run_demo = True to run a single-trial multirun and generate DVCLive artifacts.

System-monitoring note: the default config sets dvc_plugin.monitor_system=false,
so CPU and memory traces are not collected unless you override monitor_system=true.

In [13]:
run_demo = False
timeout_seconds = 300

if run_demo:
    if not DECKARD_CMD:
        raise FileNotFoundError("deckard CLI not found on PATH")
    env = os.environ.copy()
    env["DECKARD_CONFIG_DIR"] = CONFIG_DIR.as_posix()
    env.setdefault("DECKARD_TEST_MAX_SAMPLES", "100")

    result = subprocess.run(
        demo_cmd,
        cwd=PROJECT_ROOT.as_posix(),
        env=env,
        capture_output=True,
        text=True,
        check=False,
        timeout=timeout_seconds,
    )
    print("return_code:", result.returncode)
    if result.returncode != 0:
        print(result.stdout)
        print(result.stderr)
    assert result.returncode == 0, "Demo optimize run failed"
else:
    print("Set run_demo=True to execute the bounded DVCLive demo run.")

Set run_demo=True to execute the bounded DVCLive demo run.


In [14]:
monitor_paths = {
    "build_dir": BUILD_DIR,
    "dvclive_dir": BUILD_DIR / "dvclive",
    "optuna_db": BUILD_DIR / "optuna.db",
    "dvc_yaml": NOTEBOOK_DIR / "dvc.yaml",
    "dvc_lock": NOTEBOOK_DIR / "dvc.lock",
}

for name, path in monitor_paths.items():
    print(f"{name}: exists={path.exists()} path={path}")

if monitor_paths["dvclive_dir"].exists():
    dvclive_files = [p for p in sorted(monitor_paths["dvclive_dir"].rglob("*")) if p.is_file()]
    print("\nDVCLive files:")
    for p in dvclive_files[:60]:
        print("-", p.relative_to(BUILD_DIR))
    if len(dvclive_files) > 60:
        print(f"... ({len(dvclive_files) - 60} more files)")

    system_metric_candidates = [
        p
        for p in dvclive_files
        if "system" in p.name.lower() or "cpu" in p.name.lower() or "memory" in p.name.lower()
    ]
    print("\nSystem-monitoring files found:", len(system_metric_candidates))
    for p in system_metric_candidates[:20]:
        print("-", p.relative_to(BUILD_DIR))

build_dir: exists=True path=/Users/c.meyers/Documents/deckard/docs/notebooks/build/dvclivc
dvclive_dir: exists=False path=/Users/c.meyers/Documents/deckard/docs/notebooks/build/dvclivc/dvclive
optuna_db: exists=False path=/Users/c.meyers/Documents/deckard/docs/notebooks/build/dvclivc/optuna.db
dvc_yaml: exists=True path=/Users/c.meyers/Documents/deckard/docs/notebooks/dvc.yaml
dvc_lock: exists=True path=/Users/c.meyers/Documents/deckard/docs/notebooks/dvc.lock


## 4) Inspect dvc.yaml Notebook Monitoring Stage

This section inspects the notebook pipeline definition and highlights the entry for this notebook.
That stage shows how execution is tracked by DVC (deps, outs, metrics, and plots).

In [15]:
dvc_yaml_path = NOTEBOOK_DIR / "dvc.yaml"
if not dvc_yaml_path.exists():
    raise FileNotFoundError(f"Missing {dvc_yaml_path}")

dvc_cfg = OmegaConf.load(dvc_yaml_path.as_posix())
stages_cfg = dvc_cfg.get("stages", {}) if dvc_cfg is not None else {}
stage_keys = list(stages_cfg.keys()) if hasattr(stages_cfg, "keys") else []

print("dvc stage count:", len(stage_keys))
print("candidate notebook stages:")
for name in sorted([k for k in stage_keys if "notebook" in str(k)]):
    print("-", name)

target_stage_names = [name for name in stage_keys if "dvclive" in str(name)]
if not target_stage_names:
    target_stage_names = [name for name in stage_keys if "dvc" in str(name) and "notebook" in str(name)]

if target_stage_names:
    stage_name = sorted(target_stage_names)[0]
    stage_cfg = OmegaConf.to_container(stages_cfg[stage_name], resolve=False, throw_on_missing=False)
    print("\nselected_stage:", stage_name)
    print(json.dumps(stage_cfg, indent=2, default=str))
else:
    print("\nNo stage referencing dvclive was found yet in docs/notebooks/dvc.yaml.")
    print("Add a notebook_dvclive stage to include this notebook in DVC monitoring.")

dvc stage count: 17
candidate notebook stages:
- notebook_anjana
- notebook_art_attacks
- notebook_art_defenses
- notebook_artifacts
- notebook_deckard
- notebook_detector
- notebook_dvc
- notebook_fairlearn
- notebook_hydra
- notebook_lifelines
- notebook_optimize
- notebook_optuna
- notebook_pytorch
- notebook_scoring
- notebook_seaborn
- notebook_sklearn
- notebook_yellowbrick

selected_stage: notebook_dvc
{
  "cmd": "jupyter nbconvert --to notebook --execute --inplace dvc.ipynb",
  "deps": [
    "dvc.ipynb",
    "../../deckard/experiment/",
    "../../deckard/layers/",
    "../../deckard/file.py",
    "../../deckard/utils.py",
    "../../examples/sklearn/config/default.yaml",
    "../../examples/sklearn/config/files/default.yaml",
    "../../examples/sklearn/config/attack/hsj.yaml",
    "../../examples/sklearn/config/defense/class-labels.yaml",
    "../../examples/sklearn/config/plot/"
  ]
}


## 5) Inspect params.yaml Inputs and Run Materialization

This section compares parameter sources used by the default experiment:
- baseline experiment settings from examples/sklearn/config/default.yaml
- generated params.yaml files from prior runs under the notebook build directory

If no generated params.yaml files exist yet, run the demo cell first.

In [16]:
default_yaml_path = CONFIG_DIR / "default.yaml"
default_cfg = OmegaConf.load(default_yaml_path.as_posix())
default_raw = OmegaConf.to_container(default_cfg, resolve=False, throw_on_missing=False)
if not isinstance(default_raw, dict):
    default_raw = {}

hydra_cfg = default_raw.get("hydra", {}) if isinstance(default_raw.get("hydra", {}), dict) else {}
sweeper_cfg = hydra_cfg.get("sweeper", {}) if isinstance(hydra_cfg.get("sweeper", {}), dict) else {}

default_snapshot = {
    "optimizers": list(default_cfg.get("optimizers", [])),
    "directions": list(default_cfg.get("directions", [])),
    "dvclive_enabled": bool(default_cfg.get("dvclive_enabled", False)),
    "pruning_enabled": bool(default_cfg.get("pruning_enabled", False)),
    "study_name_template": sweeper_cfg.get("stuådy_name"),
    "storage_template": sweeper_cfg.get("storage"),
}

print("Default config snapshot:")
print(json.dumps(default_snapshot, indent=2, default=str))

generated_params = sorted(BUILD_DIR.rglob("params.yaml"))
print("\nGenerated params.yaml files:", len(generated_params))
for p in generated_params[:20]:
    print("-", p.relative_to(PROJECT_ROOT))

if generated_params:
    selected = generated_params[0]
    generated_cfg = OmegaConf.load(selected.as_posix())
    generated_snapshot = {
        "path": selected.as_posix(),
        "experiment_name": generated_cfg.get("experiment_name"),
        "stage": generated_cfg.get("stage"),
        "score_mode": generated_cfg.get("score_mode"),
        "optimizers": list(generated_cfg.get("optimizers", [])) if generated_cfg.get("optimizers") is not None else [],
        "directions": list(generated_cfg.get("directions", [])) if generated_cfg.get("directions") is not None else [],
    }
    print("\nSample generated params snapshot:")
    print(json.dumps(generated_snapshot, indent=2, default=str))

Default config snapshot:
{
  "optimizers": [
    "accuracy",
    "evasion_accuracy",
    "attack_generation_time"
  ],
  "directions": [
    "maximize",
    "maximize",
    "maximize"
  ],
  "dvclive_enabled": true,
  "pruning_enabled": true,
  "study_name_template": null,
  "storage_template": "sqlite:///optuna.db"
}

Generated params.yaml files: 0
